# Hari 21 — Mini-Project Penutup Minggu 3: Keputusan Model Final

**Semua bukti yang terkumpul sejauh ini:**

| | MAE Single-split | MAE CV (mean) | CV (std) | Trend Accuracy |
|---|---|---|---|---|
| Baseline naive | 12.953 | – | – | belum dihitung |
| Linear Regression | 7.806,56 | 73.013,00 | 105.471,04 | 62,07% |
| Random Forest (default) | 8.759,11 | 56.829,84 | 36.824,76 | 37,93% |
| Random Forest (tuned) | 8.456,32 | 53.165 (best CV score) | belum dihitung | belum dihitung |

Ada 3 kekosongan yang perlu dilengkapi hari ini sebelum bisa mengambil keputusan yang solid: trend accuracy baseline, trend accuracy RF tuned, dan CV std RF tuned. Setelah itu, baru kita susun kerangka keputusan resmi.

In [1]:
# Cell ini sudah lengkap — memuat ulang semua data dan melatih ulang 3 model (notebook mandiri).

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error

X_train = pd.read_csv("X_train.csv", index_col=0, parse_dates=True)
X_test = pd.read_csv("X_test.csv", index_col=0, parse_dates=True)
X_train_scaled = pd.read_csv("X_train_scaled.csv", index_col=0, parse_dates=True)
X_test_scaled = pd.read_csv("X_test_scaled.csv", index_col=0, parse_dates=True)
y_train = pd.read_csv("y_train.csv", index_col=0, parse_dates=True).iloc[:, 0]
y_test = pd.read_csv("y_test.csv", index_col=0, parse_dates=True).iloc[:, 0]

df_fitur = pd.read_csv("dataset_siap_modeling.csv", index_col=0, parse_dates=True)
X_full = df_fitur.drop(columns=["target_minggu_depan"])
y_full = df_fitur["target_minggu_depan"]

kasus_asli = pd.read_csv("dataset_bersih_minggu2.csv", index_col=0, parse_dates=True)["kasus_baru_mingguan"]
current_actual_test = kasus_asli.loc[X_test.index]

model_linear = LinearRegression().fit(X_train_scaled, y_train)
model_rf_default = RandomForestRegressor(random_state=42).fit(X_train, y_train)
model_rf_tuned = RandomForestRegressor(random_state=42, max_depth=7, min_samples_leaf=2).fit(X_train, y_train)

pred_linear = model_linear.predict(X_test_scaled)
pred_rf_default = model_rf_default.predict(X_test)
pred_rf_tuned = model_rf_tuned.predict(X_test)

print("3 model berhasil dilatih ulang: Linear Regression, RF default, RF tuned.")

3 model berhasil dilatih ulang: Linear Regression, RF default, RF tuned.


## Melengkapi Kekosongan #1: Trend Accuracy untuk RF Tuned dan Baseline

In [2]:
# Cell ini sudah lengkap — fungsi label_tren yang sama dari Hari 18.

def label_tren(nilai_depan, nilai_sekarang, ambang=0.05):
    perubahan = (nilai_depan - nilai_sekarang) / nilai_sekarang
    if perubahan > ambang:
        return "Naik"
    elif perubahan < -ambang:
        return "Turun"
    else:
        return "Stabil"

tren_aktual = [label_tren(y_test.iloc[i], current_actual_test.iloc[i]) for i in range(len(y_test))]

In [3]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Buat tren_prediksi_rf_tuned: list label_tren(pred_rf_tuned[i], current_actual_test.iloc[i])
#    untuk i dalam range(len(pred_rf_tuned)) — sama polanya seperti Hari 18
# 2. Hitung trend_accuracy_rf_tuned: proporsi tren_prediksi_rf_tuned yang sama dengan tren_aktual
# 3. Untuk BASELINE: prediksi baseline naive selalu "kasus minggu depan = kasus minggu ini"
#    (tidak ada perubahan sama sekali) — artinya label_tren-nya PASTI selalu "Stabil" untuk
#    semua baris (karena perubahan = 0% setiap saat). 
#    Buat tren_prediksi_baseline: list berisi string "Stabil" sebanyak len(y_test) kali (hint: ["Stabil"] * len(y_test))
# 4. Hitung trend_accuracy_baseline: proporsi tren_prediksi_baseline yang sama dengan tren_aktual
# 5. Cetak ketiga trend accuracy: RF tuned, dan baseline. Apa yang kamu sadari soal
#    trend accuracy baseline? (baseline secara STRUKTURAL tidak akan pernah bisa
#    memprediksi "Naik" atau "Turun" — ini alasan mendasar kenapa proyek ini butuh ML sama sekali, bukan cuma soal MAE)
# Tulis kode kamu di bawah ini:
tren_prediksi_rf_tuned = [label_tren(pred_rf_tuned[i], current_actual_test.iloc[i])
                          for i in range(len(pred_rf_tuned))]
tren_accuracy_tuned = (pd.Series(tren_prediksi_rf_tuned) == pd.Series(tren_aktual)).mean()
tren_prediksi_baseline = ["Stabil"] * len(y_test)
tren_accuracy_baseline = (pd.Series(tren_prediksi_baseline) == pd.Series(tren_aktual)).mean()
print(f"Trend Acuracy Random Foest tuned: {tren_accuracy_tuned * 100:.2f}%")
print(f"Trend Accuracy baseline: {tren_accuracy_baseline * 100:.2f}%")

Trend Acuracy Random Foest tuned: 31.03%
Trend Accuracy baseline: 6.90%


## Melengkapi Kekosongan #2: CV Std untuk RF Tuned

In [4]:
# Cell ini sudah lengkap.
tscv = TimeSeriesSplit(n_splits=5)
scores_rf_tuned = -cross_val_score(
    RandomForestRegressor(random_state=42, max_depth=7, min_samples_leaf=2),
    X_full, y_full, cv=tscv, scoring="neg_mean_absolute_error"
)
print(f"MAE CV per fold (RF tuned): {scores_rf_tuned}")
print(f"Rata-rata: {scores_rf_tuned.mean():,.2f} | Std: {scores_rf_tuned.std():,.2f}")

MAE CV per fold (RF tuned): [ 21050.37662422  71878.12293892 110008.70670117  53903.50926731
   8983.45335695]
Rata-rata: 53,164.83 | Std: 36,236.15


## Tabel Perbandingan Final (Lengkap)

In [8]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# Susun satu DataFrame final berisi SEMUA angka yang sudah terkumpul (dari notebook ini
# dan hasil Hari 16-20), dengan baris = nama model, dan
# kolom: ["MAE Single-split", "MAE CV Mean", "MAE CV Std", "Trend Accuracy (%)"]
# Nilai yang perlu kamu isi manual (angka tetap dari hari-hari sebelumnya):
# - Baseline: [12953, np.nan, np.nan, trend_accuracy_baseline*100]
# - Linear Regression: [7806.56, 73013.00, 105471.04, 62.07]
# - Random Forest (default): [8759.11, 56829.84, 36824.76, 37.93]
# - Random Forest (tuned): [8456.32, 53165, scores_rf_tuned.std(), trend_accuracy_rf_tuned*100]
# Cetak tabelnya.
# Tulis kode kamu di bawah ini:
Baseline = [12953, np.nan, np.nan, tren_accuracy_baseline*100]
Linear_Regression = [7806.56, 73013.00, 105471.04, 62.07]
RF_default = [8759.11, 56829.84, 36824.76, 37.93]
RF_tuned = [8456.32, 53165, scores_rf_tuned.std(), tren_accuracy_tuned*100]
data = {
    "MAE Single-split":[Baseline[0], Linear_Regression[0], RF_default[0], RF_tuned[0]],
    "MAE CV Mean":[Baseline[1], Linear_Regression[1], RF_default[1], RF_tuned[1]],
    "MAE CV Std":[Baseline[2], Linear_Regression[2], RF_default[2], RF_tuned[2]],
    "Trend Accuracy (%)":[Baseline[3], Linear_Regression[3], RF_default[3], RF_tuned[3]]
}
dataframe_final = pd.DataFrame(data, index=["Baseline", "Linear Regression", 
                                "Random Forest (default)", "Random Forest (tuned)"])
print(dataframe_final)

                         MAE Single-split  MAE CV Mean     MAE CV Std  \
Baseline                         12953.00          NaN            NaN   
Linear Regression                 7806.56     73013.00  105471.040000   
Random Forest (default)           8759.11     56829.84   36824.760000   
Random Forest (tuned)             8456.32     53165.00   36236.146196   

                         Trend Accuracy (%)  
Baseline                           6.896552  
Linear Regression                 62.070000  
Random Forest (default)           37.930000  
Random Forest (tuned)             31.034483  


| | MAE Single-split | MAE CV (mean) | CV (std) | Trend Accuracy |
|---|---|---|---|---|
| Baseline naive | 12.953 | Nan | Nan | 6.89% |
| Linear Regression | 7.806,56 | 73.013,00 | 105.471,04 | 62,07% |
| Random Forest (default) | 8.759,11 | 56.829,84 | 36.824,76 | 37,93% |
| Random Forest (tuned) | 8.456,32 | 53.165 (best CV score) | 36.236,14 | 31.03% |

## Kerangka Keputusan: Kembali ke Project Charter Hari 2

Ingat kembali **Success Criteria** yang kamu tetapkan sendiri di Hari 2:

- **Technical:** MAE model harus lebih rendah dari baseline, target perbaikan minimal 15-20%
- **Business:** output bisa diringkas jadi label sederhana (↑/↓/→) yang dipahami masyarakat awam

Dan pertimbangan tambahan yang muncul dari investigasi Hari 19-20:

- **Stabilitas:** karena model ini akan di-*deploy* (Minggu 4) dan dipakai di masa depan dengan data baru, model yang stabil di berbagai periode waktu (std kecil) lebih bisa dipercaya dibanding model yang cuma "kebetulan bagus" di satu periode test tertentu
- **Trade-off nyata:** Linear Regression unggul di angka MAE single-split & trend accuracy, tapi punya CV std yang jauh lebih besar (kurang stabil). Random Forest (terutama versi tuned) lebih stabil, tapi belum tentu menang di semua metrik

**Tidak ada jawaban "benar" tunggal di sini** — ini keputusan trade-off yang wajar terjadi di proyek data science nyata. Yang penting: keputusannya terdokumentasi dengan alasan jelas, bukan asal pilih angka MAE terkecil.

## Model Selection Memo — Isi Sendiri

Isi memo singkat ini sebagai dokumentasi resmi keputusanmu (ini bagian paling penting dari mini-project hari ini):

> **Model yang dipilih:** **Random Forest (tuned)**
>
> **Alasan utama:** (sebutkan metrik mana yang paling kamu prioritaskan dan kenapa — apakah MAE mentah, trend accuracy, atau stabilitas CV?)
> **trend accuracy & CV (std), karena lebih stabil walaupun akurasinya kalah dari Linear regression**
>
> **Trade-off yang disadari dan diterima:** (apa yang "dikorbankan" dari memilih model ini dibanding alternatifnya?)
> **metrik MAE Single-split**
>
> **Kondisi yang bisa mengubah keputusan ini di masa depan:** (misal: kalau nanti data lebih banyak terkumpul, atau kalau ternyata prioritas bisnis berubah)
> **Trend accuracy masih rendah**

## Simpan Model Final untuk Deployment

In [9]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Tentukan model_final = model_linear, ATAU model_rf_default, ATAU model_rf_tuned
#    (sesuaikan dengan keputusan di Model Selection Memo kamu di atas)
# 2. PENTING: kalau kamu pilih model_linear, ingat model itu butuh input yang SUDAH di-scale
#    (pakai X_test_scaled, bukan X_test mentah) — kalau pilih salah satu Random Forest,
#    pakai data mentah. Ini perlu dicatat supaya tidak salah pakai nanti pas deployment.
# 3. Simpan model_final ke file dengan joblib.dump(model_final, "model_final.pkl")
# 4. Cetak konfirmasi nama model yang disimpan, dan catat juga (print) apakah model ini
#    butuh data ter-scale atau tidak — supaya jelas dipakai di Hari 25 (Streamlit)
# Tulis kode kamu di bawah ini:
model_final = model_rf_tuned
joblib.dump(model_final, "model_final.pkl")
print("Model final yang disimpan: Random Forest tuned")
print("Model ini tidak membutuhkan data ter-scale.")

Model final yang disimpan: Random Forest tuned
Model ini tidak membutuhkan data ter-scale.


## Modeling Summary Report — Minggu 3

**Proyek:** Prediksi Tren Kasus COVID-19 Mingguan Indonesia

**Model yang dicoba:**
- Baseline naive forecast (Hari 2) — tidak pernah bisa memprediksi "Naik"/"Turun", secara struktural selalu "Stabil"
- Linear Regression (Hari 16) — MAE single-split terbaik & trend accuracy tertinggi, tapi CV mengungkap ketidakstabilan akibat multikolinearitas antar fitur lag
- Random Forest default (Hari 17) — lebih stabil antar periode waktu, tapi kalah di MAE single-split dan trend accuracy
- Random Forest tuned (Hari 19-20, `max_depth=7, min_samples_leaf=2`) — kompromi antara akurasi dan stabilitas

**Temuan metodologis penting sepanjang Minggu 3:**
- Fitur `lag_1`, `lag_2`, `lag_3`, `rolling_mean_4w` berkorelasi sangat tinggi (multikolinearitas) — koefisien Linear Regression individual tidak bisa dipercaya sebagai ukuran "importance", tapi Random Forest mengonfirmasi `lag_1` memang paling dominan
- Single train-test split bisa menyesatkan untuk data sekecil ini — cross-validation dengan `TimeSeriesSplit` mengungkap gambaran yang berbeda (bahkan terbalik untuk urutan model terbaik)
- Ada dua kriteria evaluasi yang tidak selalu sejalan: akurasi teknis (MAE) vs kebutuhan bisnis (trend accuracy) vs keandalan jangka panjang (stabilitas CV)

**Model final:** Random forest (tuned)

**Siap untuk Minggu 4 (Evaluation & Deployment):** `model_final.pkl` tersimpan dan siap dipakai di aplikasi Streamlit

## Refleksi Hari 21

> 1. Model apa yang kamu pilih sebagai final, dan seberapa yakin kamu dengan keputusan itu (skala 1-5)? → **Random Forest (tuned), 4**
> 2. Dari seluruh Minggu 3, pelajaran apa yang paling mengubah cara pandangmu terhadap "model mana yang terbaik"? → **Hari 21 Mini Project model final**
> 3. Kalau kamu harus melanjutkan proyek ini di luar 30 hari (bukan wajib, cuma refleksi), eksperimen apa yang paling ingin kamu coba duluan? (misal: fitur "lag-0" dari Hari 18, atau coba Ridge Regression untuk redam multikolinearitas) → **fitur "lag-0"**

---
### Selanjutnya: Minggu 4 (Hari 22-30) — Evaluation & Deployment

- **Hari 22**: Evaluation phase resmi CRISP-DM — review apakah keseluruhan proyek (bukan cuma model) sudah menjawab business question dari Hari 2
- **Hari 23**: Interpretasi model final lebih dalam (feature importance/koefisien final)
- **Hari 24**: Konsep deployment, load `model_final.pkl` dan uji coba prediksi manual
- **Hari 25-26**: Bangun aplikasi Streamlit — input tanggal/data terbaru, output prediksi + label tren
- **Hari 27-30**: Proyek akhir & dokumentasi penutup 30 hari